# Resultados cuantizados: varias corridas, varianza e intervalos de confianza

Esta notebook carga las corridas en `benchmarks/results_quantized/`, `results_quantized-run1/` y `results_quantized-run2/`
(más el baseline FP16 de `benchmarks/results/`) y calcula intervalos de confianza (IC) del 95 %.

**Hay dos fuentes de incertidumbre distintas y cada una se trata por separado:**

1. **Accuracy (calidad del modelo).** La evaluación es determinista (greedy, sin sampling), así que la
   accuracy sale *idéntica* en todas las corridas: la varianza entre corridas es 0 (la sección 2 lo verifica).
   Eso no significa que no haya incertidumbre. La incertidumbre real viene de que el benchmark es una
   **muestra finita de preguntas**. La pregunta es: *si tomáramos otras N preguntas del mismo tipo, ¿cuánto
   cambiaría la accuracy?* lm-eval ya reporta ese error estándar (`stderr`, que es el SE binomial
   $\sqrt{p(1-p)/n}$). El IC 95 % es $\hat p \pm 1.96\,\mathrm{SE}$.
2. **Tiempo / throughput / memoria (rendimiento).** Esto sí varía entre corridas (ruido del sistema,
   caché, temperatura de la GPU...). Con $k$ corridas se calcula la media, el desvío estándar y un IC con la
   distribución *t de Student*: $\bar x \pm t_{0.975,\,k-1}\, s/\sqrt{k}$. Con solo 2 o 3 corridas el
   factor $t$ es grande (12.7 con k=2, 4.30 con k=3), así que los IC salen anchos. Es lo correcto: con pocas
   repeticiones no podemos estar muy seguros.

In [ ]:
from pathlib import Path
import json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy import stats
    def t_crit(df, conf=0.95):
        return float(stats.t.ppf(0.5 + conf / 2, df))
except ImportError:  # fallback sin scipy (solo IC 95 %)
    _T95 = {1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776, 5: 2.571, 6: 2.447, 7: 2.365, 8: 2.306, 9: 2.262, 10: 2.228}
    def t_crit(df, conf=0.95):
        assert conf == 0.95, "sin scipy solo está soportado conf=0.95"
        return _T95.get(df, 1.96)

Z95 = 1.96

RUN_DIRS = {
    "run0": Path("benchmarks/results_quantized/raw"),
    "run1": Path("benchmarks/results_quantized-run1/raw"),
    "run2": Path("benchmarks/results_quantized-run2/raw"),
}
# Baseline FP16 viejo (una sola corrida): solo se usa si el base no está en RUN_DIRS. None = no usarlo.
BASELINE_FILE = Path("benchmarks/results/raw/meta-llama_Llama-3.1-8B.json")

# Color fijo por método (sigue a la entidad, no al orden en el gráfico)
QUANT_ORDER = ["FP16 (baseline)", "AWQ", "RTN", "NF4 (bnb)"]
QUANT_COLORS = {
    "FP16 (baseline)": "#2a78d6",
    "AWQ":             "#eb6834",
    "RTN":             "#1baf7a",
    "NF4 (bnb)":       "#eda100",
}

# Métrica "principal" de cada benchmark
SCORE_METRIC = {"sqac": "f1"}   # el resto usa "acc"

print("cwd:", Path.cwd())
for name, d in RUN_DIRS.items():
    print(f"{name}: {d}  ->", sorted(p.name for p in d.glob('*.json')))

## 1. Carga de los JSON a dos tablas largas: `scores` (métricas) y `perf` (rendimiento)

In [ ]:
def quant_label(path: Path) -> str:
    n = path.name.lower()
    if "awq" in n: return "AWQ"
    if "rtn" in n: return "RTN"
    if "nf4" in n: return "NF4 (bnb)"
    return "FP16 (baseline)"

def to_float(v):
    try:
        return float(v)
    except (TypeError, ValueError):   # "N/A", None
        return np.nan

PERF_KEYS = ["load_time_s", "ttft_ms", "prefill_tok_per_s", "decode_tok_per_s", "e2e_tok_per_s",
             "load_peak_mem_device_gb", "perf_peak_mem_alloc_gb", "perf_peak_mem_device_gb"]

files = [(run, p) for run, d in RUN_DIRS.items() for p in sorted(d.glob("*.json"))]
# El baseline viejo (una sola corrida) solo se usa si el base todavía no está en las corridas
base_in_runs = any(quant_label(p) == "FP16 (baseline)" for _, p in files)
if not base_in_runs and BASELINE_FILE is not None and BASELINE_FILE.exists():
    files.append(("baseline", BASELINE_FILE))
print("baseline:", "desde las corridas" if base_in_runs else BASELINE_FILE)

score_rows, perf_rows, time_rows = [], [], []
for run, path in files:
    data = json.loads(path.read_text())
    quant = quant_label(path)
    sample_len = {r["task"]: r["value"] for r in data["rows"] if r["metric"] == "sample_len"}
    seen_tasks = set()
    for r in data["rows"]:
        task = r["task"]
        if task not in seen_tasks:
            seen_tasks.add(task)
            time_rows.append({"run": run, "quant": quant, "task": task,
                              "total_time_s": to_float(r.get("total_time_s")),
                              "time_per_sample_s": to_float(r.get("time_per_sample_s"))})
        if r["metric"] == "sample_len":
            continue
        n = r.get("n_samples") or sample_len.get(task)
        score_rows.append({"run": run, "quant": quant, "task": task, "metric": r["metric"],
                           "value": to_float(r["value"]), "stderr": to_float(r.get("stderr")),
                           "n": int(n) if n else np.nan})
    perf = data.get("perf", {})
    perf_rows.append({"run": run, "quant": quant, **{k: to_float(perf.get(k)) for k in PERF_KEYS}})

scores = pd.DataFrame(score_rows)
perf_df = pd.DataFrame(perf_rows)
times = pd.DataFrame(time_rows)

# Qué hay en cada corrida (las corridas incompletas aparecen con huecos)
scores.pivot_table(index=["quant", "task"], columns="run", values="value", aggfunc="count").fillna(0).astype(int)

## 2. ¿Cambia la accuracy entre corridas?

Si `std_entre_corridas` es 0, la evaluación es determinista y repetir corridas **no** aporta información sobre la
calidad; el IC tiene que salir de la variabilidad entre preguntas (sección 3).

In [ ]:
quant_runs = scores[scores.run != "baseline"]  # el baseline viejo no es una corrida repetida
run_var = (quant_runs.groupby(["quant", "task", "metric"])["value"]
           .agg(n_corridas="count", media="mean", std_entre_corridas="std",
                min="min", max="max")
           .reset_index())
run_var

## 3. IC 95 % de la accuracy (variabilidad por muestreo de preguntas)

- Si lm-eval da `stderr` → se usa ese.
- Si no (las métricas de `sqac`), y la métrica es por pregunta 0/1 (`exact_match`, `contains`) → SE binomial $\sqrt{p(1-p)/n}$.
- Para `f1` de sqac no se puede sacar el SE solo con el promedio: haría falta el F1 **de cada pregunta**
  para hacer un *bootstrap* (re-muestrear preguntas con reemplazo muchas veces y mirar la dispersión del promedio).
  Por ahora queda sin IC.

In [ ]:
BINARY_METRICS = {"acc", "acc_norm", "exact_match", "contains"}

def se_for(row):
    if not np.isnan(row["stderr"]):
        return row["stderr"]
    if row["metric"] in BINARY_METRICS and row["n"] and not np.isnan(row["n"]):
        p = row["value"]
        return math.sqrt(p * (1 - p) / row["n"])
    return np.nan

# Una fila por (método, tarea, métrica): como el valor no cambia entre corridas, se toma la media
acc = (scores.groupby(["quant", "task", "metric"])
       .agg(value=("value", "mean"), stderr=("stderr", "mean"), n=("n", "max"),
            n_corridas=("run", "count"))
       .reset_index())
acc["se"] = acc.apply(se_for, axis=1)
acc["ci_low"] = acc["value"] - Z95 * acc["se"]
acc["ci_high"] = acc["value"] + Z95 * acc["se"]
acc["ci_half"] = Z95 * acc["se"]

acc["quant"] = pd.Categorical(acc["quant"], QUANT_ORDER, ordered=True)
acc = acc.sort_values(["task", "metric", "quant"]).reset_index(drop=True)
acc.round(4)

In [ ]:
# Tabla compacta "valor ± IC" con la métrica principal de cada benchmark (lista para la tesis)
main = acc[acc.apply(lambda r: r["metric"] == SCORE_METRIC.get(r["task"], "acc"), axis=1)].copy()

def fmt(r):
    if np.isnan(r["ci_half"]):
        return f"{r['value']:.3f}"
    return f"{r['value']:.3f} ± {r['ci_half']:.3f}"

main["valor ± IC95"] = main.apply(fmt, axis=1)
main.pivot(index="quant", columns="task", values="valor ± IC95").reindex(
    [q for q in QUANT_ORDER if q in set(main["quant"])])

In [ ]:
def grouped_bars(ax, df, group_col, value_col, err_col, groups, ylabel, title, ylim=None, fmt="{:.2f}", log=False):
    # Barras agrupadas: un grupo por `group_col`, una barra por método, error bar = IC 95 %
    quants = [q for q in QUANT_ORDER if q in set(df["quant"])]
    x = np.arange(len(groups))
    width = 0.8 / len(quants)
    for i, q in enumerate(quants):
        sub = df[df["quant"] == q].set_index(group_col).reindex(groups)
        vals = sub[value_col].astype(float).values
        errs = sub[err_col].astype(float).values
        offset = (i - (len(quants) - 1) / 2) * width
        ax.bar(x + offset, vals, width * 0.92, label=q, color=QUANT_COLORS[q])
        errs = np.nan_to_num(errs)
        lower = np.minimum(errs, vals * 0.95) if log else errs   # en escala log no se puede bajar de 0
        ax.errorbar(x + offset, vals, yerr=[lower, errs], fmt="none",
                    ecolor="#333333", elinewidth=1.2, capsize=3)
        for xi, v, e in zip(x + offset, vals, errs):
            if not np.isnan(v):
                y_txt = (v + e) * 1.08 if log else v + e + (ylim[1] if ylim else np.nanmax(vals)) * 0.01
                ax.text(xi, y_txt, fmt.format(v), ha="center", va="bottom", fontsize=7, color="#333333")
    if log: ax.set_yscale("log")
    ax.set_xticks(x)
    ax.set_xticklabels(groups)
    if ylim: ax.set_ylim(*ylim)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.3)
    ax.set_axisbelow(True)
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    ax.legend(title="Método", loc="upper center", bbox_to_anchor=(0.5, -0.1),
              ncol=len(quants), frameon=False)

main["bench"] = main.apply(lambda r: f"{r['task']} ({r['metric']})", axis=1)
benches = sorted(main["bench"].unique())

fig, ax = plt.subplots(figsize=(11, 5.5))
grouped_bars(ax, main, "bench", "value", "ci_half", benches,
             "Score", "Score por benchmark y método (barras = IC 95 % por muestreo de preguntas)",
             ylim=(0, 1))
plt.tight_layout()
plt.show()

### Gráfico de "forest plot" (diferencia contra el baseline)

Lo que realmente interesa es: *¿la cuantización empeora el modelo?* Para eso se grafica la diferencia
$\Delta = \text{score}_{\text{cuant}} - \text{score}_{\text{FP16}}$ con su IC. Si el IC **cruza el 0**, la diferencia
no es estadísticamente significativa con este tamaño de benchmark.

Aproximación: $\mathrm{SE}_\Delta = \sqrt{\mathrm{SE}_1^2 + \mathrm{SE}_2^2}$. Como ambos modelos responden las *mismas*
preguntas, esta fórmula es conservadora (el IC real, pareado, es más angosto). Con las predicciones por pregunta
se podría hacer un test pareado (McNemar o bootstrap pareado).

In [ ]:
base = main[main["quant"] == "FP16 (baseline)"].set_index("bench")
diff = main[main["quant"] != "FP16 (baseline)"].copy()
diff = diff[diff["bench"].isin(base.index)]
diff["delta"] = diff["value"].values - base.loc[diff["bench"], "value"].values
diff["se_delta"] = np.sqrt(diff["se"].values ** 2 + base.loc[diff["bench"], "se"].values ** 2)
diff["delta_ci_half"] = Z95 * diff["se_delta"]
diff["significativo"] = (diff["delta"].abs() > diff["delta_ci_half"])

if diff.empty:
    print("No hay baseline para comparar.")
else:
    display(diff[["bench", "quant", "delta", "delta_ci_half", "significativo"]].round(4))

    benches_d = sorted(diff["bench"].unique())
    quants_d = [q for q in QUANT_ORDER if q in set(diff["quant"])]
    fig, ax = plt.subplots(figsize=(9, 0.9 * len(benches_d) * len(quants_d) / 2 + 2))
    y = np.arange(len(benches_d))
    h = 0.8 / len(quants_d)
    for i, q in enumerate(quants_d):
        sub = diff[diff["quant"] == q].set_index("bench").reindex(benches_d)
        yy = y + (i - (len(quants_d) - 1) / 2) * h
        ax.errorbar(sub["delta"], yy, xerr=sub["delta_ci_half"], fmt="o", ms=8,
                    color=QUANT_COLORS[q], ecolor=QUANT_COLORS[q], elinewidth=2, capsize=4,
                    markeredgecolor="white", markeredgewidth=1.5, label=q)
    ax.axvline(0, color="#555555", lw=1, ls="--")
    ax.set_yticks(y)
    ax.set_yticklabels(benches_d)
    ax.invert_yaxis()
    ax.set_xlabel("Δ score vs FP16 (IC 95 %)")
    ax.set_title("Diferencia contra el baseline: si la barra cruza 0, no es significativa")
    ax.grid(axis="x", alpha=0.3)
    ax.set_axisbelow(True)
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    ax.legend(title="Método", loc="upper center", bbox_to_anchor=(0.5, -0.15),
              ncol=len(quants_d), frameon=False)
    plt.tight_layout()
    plt.show()

## 4. Rendimiento: varianza entre corridas e IC con t de Student

Acá sí hay variación entre corridas. Para cada métrica: media, desvío estándar (`std`, con ddof=1),
coeficiente de variación (`cv_% = std/media`) y IC 95 % $= \bar x \pm t_{0.975,k-1}\, s/\sqrt{k}$.
Con una sola corrida (el baseline) no se puede estimar varianza → IC vacío.

In [ ]:
def summarize(df, keys, value_cols):
    long = df.melt(id_vars=keys + ["run"], value_vars=value_cols, var_name="metric", value_name="x")
    long = long.dropna(subset=["x"])
    g = long.groupby(keys + ["metric"])["x"]
    out = g.agg(k="count", mean="mean", std="std").reset_index()
    out["cv_%"] = 100 * out["std"] / out["mean"]
    out["ci_half"] = [t_crit(k - 1) * s / math.sqrt(k) if k > 1 else np.nan
                      for k, s in zip(out["k"], out["std"])]
    out["ci_low"] = out["mean"] - out["ci_half"]
    out["ci_high"] = out["mean"] + out["ci_half"]
    out["quant"] = pd.Categorical(out["quant"], QUANT_ORDER, ordered=True)
    return out.sort_values(keys + ["metric"]).reset_index(drop=True)

perf_summary = summarize(perf_df, ["quant"], PERF_KEYS)
perf_summary.round(3)

In [ ]:
# Una métrica por gráfico (escalas distintas -> nunca dos ejes Y en el mismo gráfico)
PLOT_PERF = [
    ("prefill_tok_per_s", "tokens/s", "Prefill throughput"),
    ("decode_tok_per_s",  "tokens/s", "Decode throughput"),
    ("ttft_ms",           "ms",       "Time to first token"),
    ("perf_peak_mem_device_gb", "GB", "Memoria pico del dispositivo"),
]

fig, axes = plt.subplots(1, len(PLOT_PERF), figsize=(4.2 * len(PLOT_PERF), 4.8))
for ax, (metric, unit, title) in zip(axes, PLOT_PERF):
    sub = perf_summary[perf_summary["metric"] == metric]
    sub = sub[sub["quant"].isin(QUANT_ORDER)].sort_values("quant")
    labels = [str(q) for q in sub["quant"]]
    x = np.arange(len(sub))
    ax.bar(x, sub["mean"], 0.7, color=[QUANT_COLORS[q] for q in labels])
    ax.errorbar(x, sub["mean"], yerr=np.nan_to_num(sub["ci_half"]), fmt="none",
                ecolor="#333333", elinewidth=1.2, capsize=4)
    # cada corrida individual como punto, para ver la dispersión real
    for xi, q in zip(x, labels):
        pts = perf_df.loc[perf_df["quant"] == q, metric].dropna()
        ax.scatter(np.full(len(pts), xi) + np.linspace(-0.12, 0.12, len(pts)), pts,
                   s=18, color="#222222", zorder=3)
    for xi, (m, k) in zip(x, zip(sub["mean"], sub["k"])):
        ax.text(xi, 0, f"n={k}", ha="center", va="bottom", fontsize=7, color="white")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=20, ha="right", fontsize=8)
    ax.set_ylabel(unit)
    ax.set_title(title, fontsize=10)
    ax.grid(axis="y", alpha=0.3)
    ax.set_axisbelow(True)
    for s in ("top", "right"): ax.spines[s].set_visible(False)
fig.suptitle("Rendimiento: media ± IC 95 % (t de Student) entre corridas; puntos = corridas individuales")
plt.tight_layout()
plt.show()

In [ ]:
# Prefill vs memoria, con barras de error (IC 95 %) en ambos ejes
ps = perf_summary.set_index(["quant", "metric"])
fig, ax = plt.subplots(figsize=(8, 5.5))
for q in [q for q in QUANT_ORDER if q in set(perf_df["quant"])]:
    xm, xe = ps.loc[(q, "prefill_tok_per_s"), ["mean", "ci_half"]]
    ym, ye = ps.loc[(q, "perf_peak_mem_device_gb"), ["mean", "ci_half"]]
    ax.errorbar(xm, ym, xerr=np.nan_to_num(xe), yerr=np.nan_to_num(ye), fmt="o", ms=11,
                color=QUANT_COLORS[q], ecolor=QUANT_COLORS[q], elinewidth=2, capsize=4,
                markeredgecolor="white", markeredgewidth=2, label=q, zorder=3)
    ax.annotate(f"{xm:.0f} tok/s\n{ym:.2f} GB", (xm, ym), textcoords="offset points",
                xytext=(10, 8), fontsize=8, color="#333333")
ax.set_xlabel("Prefill throughput (tokens/s)")
ax.set_ylabel("Memoria pico del dispositivo (GB)")
ax.set_title("Prefill vs memoria pico (media ± IC 95 %)")
ax.margins(0.15)
ax.grid(alpha=0.3)
ax.set_axisbelow(True)
for s in ("top", "right"): ax.spines[s].set_visible(False)
ax.legend(title="Método", loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=4, frameon=False)
plt.tight_layout()
plt.show()

### Tiempo total de cada benchmark (varianza entre corridas)

In [ ]:
time_summary = summarize(times, ["quant", "task"], ["total_time_s"])
display(time_summary.round(2))

tasks = sorted(time_summary["task"].unique())
fig, ax = plt.subplots(figsize=(11, 5.5))
grouped_bars(ax, time_summary, "task", "mean", "ci_half", tasks,
             "Tiempo total (s, escala log)", "Tiempo por benchmark: media ± IC 95 % entre corridas",
             fmt="{:.0f}", log=True)
plt.tight_layout()
plt.show()

## Notas para la tesis

- **Accuracy:** reportar `valor ± IC95` donde el IC viene del error estándar binomial (muestreo de preguntas).
  Repetir corridas no lo achica porque la evaluación es determinista; lo que lo achicaría es un benchmark con más preguntas.
- **Rendimiento:** reportar media ± IC95 con t de Student e indicar *k* (número de corridas). Con k=2–3 los IC son
  anchos; con 5 o más corridas se vuelven mucho más útiles.
- **Comparar métodos:** mirar si el IC de la diferencia contra el baseline cruza 0 (forest plot).
- **Pendiente:** para tener IC del F1 de SQAC (y tests pareados) hay que guardar el score de cada pregunta y hacer bootstrap.